In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g": 0.2,
    "Dry_Total_g": 0.5
}

DATASET_DIR = "/kaggle/input/competitions/csiro-biomass"
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
TEST_CSV = os.path.join(DATASET_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train")
TEST_IMG_DIR = os.path.join(DATASET_DIR, "test")

print("train.csv exists:", os.path.exists(TRAIN_CSV))
print("test.csv exists:", os.path.exists(TEST_CSV))
print("train dir exists:", os.path.exists(TRAIN_IMG_DIR))
print("test dir exists:", os.path.exists(TEST_IMG_DIR))

train.csv exists: True
test.csv exists: True
train dir exists: True
test dir exists: True


In [2]:
def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]

    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target"
    ).reset_index()

    y_values = df_wide[TARGETS].values
    return df_wide, y_values


def prepare_submission(test_csv_path, predictions, image_ids):
    df_test = pd.read_csv(test_csv_path)

    pred_dict = {}
    for img_id, pred_vector in zip(image_ids, predictions):
        pred_dict[img_id] = {col: val for col, val in zip(TARGETS, pred_vector)}

    def get_pred(row):
        img_id = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        val = pred_dict.get(img_id, {}).get(target_name, 0.0)
        return max(0.0, float(val))

    df_test["target"] = df_test.apply(get_pred, axis=1)
    return df_test[["sample_id", "target"]]


def weighted_global_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    w = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])

    ybar = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    out = {}
    for i, t in enumerate(TARGETS):
        out[t] = float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
    return out

In [3]:
class ResNetExtractorV2:
    def __init__(self, device="cpu"):
        self.device = device

        # Try pretrained weights first
        try:
            weights = models.ResNet50_Weights.DEFAULT
            resnet = models.resnet50(weights=weights)
            print("Loaded pretrained ResNet50 weights.")
        except Exception as e:
            print("Could not load pretrained weights, falling back to random init:", e)
            resnet = models.resnet50(weights=None)

        self.model = nn.Sequential(*list(resnet.children())[:-1])
        self.model.to(self.device)
        self.model.eval()

        self.preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def _extract_single(self, pil_img):
        t = self.preprocess(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feat = self.model(t).flatten().cpu().numpy()
        return feat.astype(np.float32)

    def get_features(self, image_path):
        if not os.path.exists(image_path):
            print(f"Missing file: {image_path}")
            return np.zeros(4096, dtype=np.float32)

        try:
            img = Image.open(image_path).convert("RGB")

            # Original image is 2000x1000
            img_left = img.crop((0, 0, 1000, 1000))
            img_right = img.crop((1000, 0, 2000, 1000))

            feat_left = self._extract_single(img_left)
            feat_right = self._extract_single(img_right)

            # Concatenate instead of average
            feat = np.concatenate([feat_left, feat_right], axis=0)
            return feat.astype(np.float32)

        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return np.zeros(4096, dtype=np.float32)

In [4]:
train_df, y_all = load_train_data(TRAIN_CSV)

print("Train images:", len(train_df))
print("Target shape:", y_all.shape)
print(train_df.head())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

extractor = ResNetExtractorV2(device=device)

all_train_features = []
all_image_ids = []

for i, row in enumerate(train_df.itertuples(index=False), 1):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row.image_id}.jpg")
    feat = extractor.get_features(img_path)

    all_train_features.append(feat)
    all_image_ids.append(row.image_id)

    if i % 200 == 0:
        print(f"Processed {i}/{len(train_df)} train images")

X_all = np.array(all_train_features, dtype=np.float32)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

# Optional cache for faster iteration
np.save("/kaggle/working/X_all_v2.npy", X_all)
np.save("/kaggle/working/y_all_v2.npy", y_all)
print("Saved cached train arrays to /kaggle/working/")

Train images: 357
Target shape: (357, 5)
target_name      image_id              image_path  Dry_Clover_g  Dry_Dead_g  \
0            ID1011485656  train/ID1011485656.jpg        0.0000     31.9984   
1            ID1012260530  train/ID1012260530.jpg        0.0000      0.0000   
2            ID1025234388  train/ID1025234388.jpg        6.0500      0.0000   
3            ID1028611175  train/ID1028611175.jpg        0.0000     30.9703   
4            ID1035947949  train/ID1035947949.jpg        0.4343     23.2239   

target_name  Dry_Green_g  Dry_Total_g    GDM_g  
0                16.2751      48.2735  16.2750  
1                 7.6000       7.6000   7.6000  
2                 0.0000       6.0500   6.0500  
3                24.2376      55.2079  24.2376  
4                10.5261      34.1844  10.9605  
Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
Could not load pretrained weights, fall

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42
)

print("Train split:", X_tr.shape, y_tr.shape)
print("Val split:", X_val.shape, y_val.shape)

model = MultiOutputRegressor(
    Ridge(alpha=10.0, random_state=42)
)

model.fit(X_tr, y_tr)
val_preds = model.predict(X_val)

print("Validation weighted global R2:", weighted_global_r2(y_val, val_preds))
print("Validation overall R2:", r2_score(y_val, val_preds, multioutput="uniform_average"))
print("Validation RMSE per target:")
print(rmse_per_target(y_val, val_preds))

Train split: (285, 4096) (285, 5)
Val split: (72, 4096) (72, 5)
Validation weighted global R2: 0.1175663710902084
Validation overall R2: -0.12077928937686859
Validation RMSE per target:
{'Dry_Green_g': 24.21856352142984, 'Dry_Dead_g': 12.992722116342733, 'Dry_Clover_g': 12.906413628407675, 'GDM_g': 21.832259336216175, 'Dry_Total_g': 27.127759262463332}


In [6]:
final_model = MultiOutputRegressor(
    Ridge(alpha=10.0, random_state=42)
)

final_model.fit(X_all, y_all)
print("Final model trained on all training data.")

df_test = pd.read_csv(TEST_CSV)
df_test["image_id"] = df_test["sample_id"].str.split("__").str[0]
df_test_unique = df_test.drop_duplicates("image_id").copy()

print("Test images:", len(df_test_unique))
print(df_test_unique.head())

Final model trained on all training data.
Test images: 1
                    sample_id             image_path   target_name  \
0  ID1001187975__Dry_Clover_g  test/ID1001187975.jpg  Dry_Clover_g   

       image_id  
0  ID1001187975  


In [7]:
test_features = []
test_image_ids = []

for i, row in enumerate(df_test_unique.itertuples(index=False), 1):
    img_path = os.path.join(TEST_IMG_DIR, f"{row.image_id}.jpg")
    feat = extractor.get_features(img_path)

    test_features.append(feat)
    test_image_ids.append(row.image_id)

    if i % 200 == 0:
        print(f"Processed {i}/{len(df_test_unique)} test images")

X_test = np.array(test_features, dtype=np.float32)
print("X_test shape:", X_test.shape)

test_preds = final_model.predict(X_test)
print("test_preds shape:", test_preds.shape)

submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)
print(submission.head())

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission to /kaggle/working/submission.csv")
print("Working dir files:", os.listdir("/kaggle/working"))

X_test shape: (1, 4096)
test_preds shape: (1, 5)
                    sample_id     target
0  ID1001187975__Dry_Clover_g   2.101597
1    ID1001187975__Dry_Dead_g  20.649399
2   ID1001187975__Dry_Green_g   1.788631
3   ID1001187975__Dry_Total_g  24.518597
4         ID1001187975__GDM_g   3.886963
Saved submission to /kaggle/working/submission.csv
Working dir files: ['__notebook__.ipynb', 'X_all_v2.npy', 'submission.csv', 'y_all_v2.npy']
